In [ ]:
# From Bob: I uncommented this line or else I cannot query the data in block 3.
from project import icu_query

In [ ]:
# Import custom functions & libraries
from project import bucket_ecg_report_0

In [ ]:
original_query = icu_query()

In [ ]:
df = original_query
df.head()

In [ ]:
# need to bucket report_0...
# getting frequency of report_0 codes
original_ecg_report0_frequency = original_query['report_0'].value_counts()
original_ecg_report0_frequency.to_csv('sanity_outputs/original_ecg_report0_frequency.csv')

In [ ]:
# sinus rhythm, sinus tachycardia, a.fib
df['report_0'] = df['report_0'].str.lower().str.removesuffix(".").str.replace('"', '')
df = df[~df['report_0'].str.contains('warning')]
initial_drop = df['report_0'].value_counts()
initial_drop.to_csv('sanity_outputs/initial_drop.csv')
# handles: Sinus rhythm, Sinus tachycardia, Atrial fibrillation, Sinus bradycardia that had '.' suffix
# changes all to lower, drops records with 'warning' because those signals could harm our model
df = df[df['report_0'].notna()] # handles any NA values

In [ ]:
# check if any NAs, there were none
print(original_query.shape)
print(df.shape)

In [ ]:
# check mortality captured by these first four catgories ()
df_first_cats = df[df['report_0'].isin(['sinus rhythm', 'sinus tachycardia', 'atrial fibrillation', 'sinus bradycardia'])]
print(f"")
print(f"Mortalities captured by the first four report_0 buckets: {sum(df_first_cats['hospital_expire_flag'])} ({sum(df_first_cats['hospital_expire_flag']) / sum(original_query['hospital_expire_flag']):.2%})")
print(f"Total mortalities: {sum(df['hospital_expire_flag'])}")

In [ ]:
df['ecg_bucket'] = df['report_0'].apply(bucket_ecg_report_0)
# check mortality captured after bucket_ecg_report_0 function
df_cats = df[df['ecg_bucket'].isin(['normal_sinus', 'sinus_tachy', 'sinus_brady', 'afib', 'afib_rvr', 'pvc', 'pac', 'paced', 'stemi_alert', 'sinus arrhythmia', 'atrial_ectopic', 'supraventricular', 'idioventricular', 'accelerated_junctional', 'other'])]
print(f"Mortalities captured by the current buckets: {sum(df_cats['hospital_expire_flag'])} ({sum(df_cats['hospital_expire_flag']) / sum(df['hospital_expire_flag']):.2%})")
print(f"Total mortalities: {sum(df['hospital_expire_flag'])}")
post_bucket_function = df['ecg_bucket'].value_counts()
post_bucket_function.to_csv('sanity_outputs/post_bucket_function.csv')
# capturing 98.7% of mortalities in all categories != 'other'

In [ ]:
named_buckets = ['normal_sinus', 'sinus_tachy', 'sinus_brady', 'afib', 'afib_rvr', 'pvc', 'pac', 'paced', 'stemi_alert', 'atrial_ectopic', 'supraventricular', 'idioventricular', 'accelerated_junctional']
df_other = df[~df['ecg_bucket'].isin(named_buckets)]
df_other.groupby('ecg_bucket')['hospital_expire_flag'].agg(['sum', 'count', 'mean']).sort_values('sum', ascending=False)
# these columns made up 'other' not sure if we keep or omit? contains 1.3% mortalities.

In [ ]:
df['ecg_bucket'] = df['ecg_bucket'].where(df['ecg_bucket'].isin(named_buckets), 'other')
df.to_csv('sanity_outputs/report0_cleaned.csv')
# report_0 cleaned values in new column: ecg_bucket

In [ ]:
# Bob: Check columns, head, and export cleaned data to csv
print(df.columns)
display(df.head())
# df.to_csv('cleaned_data.csv')